# Effect of KL Regularisation Strength $\beta$ on Model Performance

There is a tradeoff between reconstruction and generation quality, as controlled by the $\beta$ KL regularisation strength.

We can observe from Tensorboard logs the quantitative metrics. In this notebook, let's see how $\beta$ affects the reconstruction and generation quality by inspecting the outputs visually.

## Preliminaries

The repository assumes all entry points to be run as modules from root. Since
this notebook is 2 levels deep, the below block moves us to the root directory.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

Update lines 10 and 11 to choose 2 models. We will compare the reconstructions and generations of these models.

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from arch.vae.vae import VAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path_b1 = "logs/nvae_acdc/latent-skip-clamp/pc-4-ws-6420-b-1/checkpoints/epoch=95-step=20544.ckpt"
model_path_b18 = "logs/nvae_acdc/latent-skip-clamp/pc-4-ws-6420-b-18/checkpoints/epoch=96-step=20758.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=6)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model_b1 = NVAE.load_from_checkpoint(model_path_b1)
model_b18 = NVAE.load_from_checkpoint(model_path_b18)

In [ ]:
loader_test = data_module.test_dataloader(shuffle=True)
feats = next(iter(loader_test))
feats.shape

## Reconstructions

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff
from utils.utils import show_samples

with torch.no_grad():
    model_b1.eval()
    model_b1.to(device)
    feats = feats.to(device)
    
    feats_hat_logits_b1, _, _, _, _ = model_b1(feats, test=True)
    
with torch.no_grad():
    model_b18.eval()
    model_b18.to(device)
    feats = feats.to(device)
    
    feats_hat_logits_b18, _, _, _, _ = model_b18(feats, test=True)

samples, reconstructions_b1, reconstruction_pixel_error_b1 = get_samples_and_reconstructions_pixel_diff(
    feats,
    feats_hat_logits_b1,
    return_reconstructions=True,
)

_, reconstructions_b18, reconstruction_pixel_error_b18 = get_samples_and_reconstructions_pixel_diff(
    feats,
    feats_hat_logits_b18,
    return_reconstructions=True,
)

View and compare reconstructions.

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

from utils.colourmap import GIRIDIS, GREDS

fig, ax = plt.subplots(1, 5, figsize=(30, 36))

# Column 1: Ground truth

samples = samples.cpu().float()
samples_grid = make_grid(samples, nrow=1, padding=2, normalize=True)
samples_grid = samples_grid[0]

ax[0].axis("off")
ax[0].imshow(samples_grid, cmap=GIRIDIS)
ax[0].set_title("Ground Truth", fontsize=36, pad=36)

# Column 2: NVAE reconstruction (b=1)

reconstructions_b1 = reconstructions_b1.cpu().float()
reconstructions_grid = make_grid(reconstructions_b1, nrow=1, padding=2, normalize=True)
reconstructions_grid = reconstructions_grid[0]

ax[1].axis("off")
ax[1].imshow(reconstructions_grid, cmap=GIRIDIS)
ax[1].set_title(r"$\beta = 1$" + "\n" + "Reconstruction", fontsize=36, pad=36)

# Column 3: NVAE reconstruction (b=18)

reconstructions_b18 = reconstructions_b18.cpu().float()
reconstructions_vae_grid = make_grid(reconstructions_b18, nrow=1, padding=2, normalize=True)
reconstructions_vae_grid = reconstructions_vae_grid[0]

ax[2].axis("off")
ax[2].imshow(reconstructions_vae_grid, cmap=GIRIDIS)
ax[2].set_title(r"$\beta = 18$" + "\n" + "Reconstruction", fontsize=36, pad=36)

# Column 4: NVAE reconstruction error (b=1)

reconstruction_pixel_error_b1 = reconstruction_pixel_error_b1.cpu().float()
reconstruction_pixel_error_grid = make_grid(reconstruction_pixel_error_b1, nrow=1, padding=2, normalize=True)
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid > 0
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid[0]

ax[3].axis("off")
ax[3].imshow(reconstruction_pixel_error_grid, cmap=GREDS)
ax[3].set_title(r"$\beta = 1$" + "\n" + "Incorrect Pixels", fontsize=36, pad=36)

# Column 5: NVAE reconstruction error (b=18)

reconstruction_pixel_error_b18 = reconstruction_pixel_error_b18.cpu().float()
reconstruction_pixel_error_vae_grid = make_grid(reconstruction_pixel_error_b18, nrow=1, padding=2, normalize=True)
reconstruction_pixel_error_vae_grid = reconstruction_pixel_error_vae_grid > 0
reconstruction_pixel_error_vae_grid = reconstruction_pixel_error_vae_grid[0]

ax[4].axis("off")
ax[4].imshow(reconstruction_pixel_error_vae_grid, cmap=GREDS)
ax[4].set_title(r"$\beta = 18$" + "\n" + "Incorrect Pixels", fontsize=36, pad=36)

## Generations

In [ ]:
# Update this line to choose which model to view
model = model_b1

with torch.no_grad():
    model.eval()
    model.to(device)
    
    x_fake = model.decoder.generate(num_samples=24, device=device)
    feats_fake = model.conditional_coder(x_fake)

# Discretise probabilistic map then view generations
generations = torch.argmax(feats_fake, dim=1).unsqueeze(1)
show_samples(generations, rgb=False, ncol=6, figsize=(24, 16))